<center><h1>Disi_Sinan_HW3</h1></center>
<br>
<br>

Name: Sinan Disi
<br>
Github Username: Sinan-Disi
<br>
USC ID: 8053361335

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
from scipy.stats import bootstrap

Get the AReM Data Set

In [2]:
with zipfile.ZipFile("../data/AReM.zip", "r") as zip_ref:
    zip_ref.extractall("../data")

data_path = Path("../data/AReM")
print("CSV files found:", len(list(data_path.rglob("*.csv"))))

CSV files found: 88


### (b) Test and Train Data

In [3]:
train_files = []
test_files = []

for activity_folder in data_path.iterdir():
    if activity_folder.is_dir():
        activity = activity_folder.name

        for file in activity_folder.glob("dataset*.csv"):
            dataset_num = int(file.stem.replace("dataset", ""))

            if activity in ["bending1", "bending2"]:
                if dataset_num <= 2:
                    test_files.append(file)
                else:
                    train_files.append(file)
            else:
                if dataset_num <= 3:
                    test_files.append(file)
                else:
                    train_files.append(file)

print("Training files:", len(train_files))
print("Test files:", len(test_files))

Training files: 69
Test files: 19


### (c) Feature Extraction

#### i. Research

Some common time domain features are minimum, maximum, mean, median,
standard deviation, variance, range, first quartile, third quartile, and
interquartile range.

These features summarize the main characteristics of a time series without
using every individual data point.

Source: [sktime documentation - SummaryTransformer](https://www.sktime.net/docs/examples/transformers/)

#### ii. Extraction

In [4]:
columns = [
    "time", "avg_rss12", "var_rss12",
    "avg_rss13", "var_rss13",
    "avg_rss23", "var_rss23"]

feature_rows = []
for file in sorted(data_path.rglob("dataset*.csv")):

    df = pd.read_csv(
        file,
        comment="#",
        header=None,
        names=columns,
        sep=r"[,\s]+",
        engine="python",
        usecols=range(7))

    row = {}
    for i, column in enumerate(columns[1:], start=1):
        row[f"min{i}"] = df[column].min()
        row[f"max{i}"] = df[column].max()
        row[f"mean{i}"] = df[column].mean()
        row[f"median{i}"] = df[column].median()
        row[f"std{i}"] = df[column].std()
        row[f"1st_quart{i}"] = df[column].quantile(0.25)
        row[f"3rd_quart{i}"] = df[column].quantile(0.75)

    feature_rows.append(row)

features = pd.DataFrame(feature_rows)

print(features.shape)
features.head()

(88, 42)


,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,...,std5,1st_quart5,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6
0,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,0.0,1.30,0.358604,...,2.188449,33.0000,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30
1,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,0.0,1.22,0.372437,...,1.995255,32.0000,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30
2,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,0.426250,...,1.999604,35.3625,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94
3,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,0.696042,...,3.849448,30.4575,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00
4,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,0.535979,...,2.411026,28.4575,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50


#### iii. Standard Deviation

In [5]:
feature_std = features.std()

bootstrap_results = []

for column in features.columns:
    result = bootstrap(
        (features[column].values,),
        lambda x: np.std(x, ddof=1),
        confidence_level=0.90,
        n_resamples=1000,
        random_state=42,
        method="percentile")

    bootstrap_results.append([
        feature_std[column],
        result.confidence_interval.low,
        result.confidence_interval.high])

std_results = pd.DataFrame(
    bootstrap_results,
    index=features.columns,
    columns=["Standard Deviation", "90% CI Lower", "90% CI Upper"])

std_results

,Standard Deviation,90% CI Lower,90% CI Upper
min1,9.569975,8.241109,10.782064
max1,4.394362,3.341984,5.269646
mean1,5.335718,4.677349,5.866294
median1,5.440054,4.757716,6.027372
std1,1.772153,1.569312,1.951916
1st_quart1,6.153590,5.542305,6.643123
3rd_quart1,5.138925,4.287237,5.825419
min2,0.000000,0.000000,0.000000
max2,5.062729,4.621948,5.415051
mean2,1.574164,1.409308,1.707793


#### iv. Select Features

I would choose mean, standard deviation, and maximum.

Mean shows the average level of the signal, standard deviation shows how much it changes, and maximum shows the highest value it reaches.

## 2. ISLR 3.7.4

### (a) Linear Train

I would expect the cubic model to have a lower or equal training RSS because it is more flexible and can fit the training data more closely.


### (b) Linear Test

I would expect the linear model to have a lower test RSS because the real relationship is linear. The extra cubic terms are not needed and may fit noise, which can increase variance.

### (c) Not Linear Train

The cubic model should still have a lower or equal training RSS since it has more flexibility than the linear model.


### (d) Not Linear Testing

We cannot say for sure which one will have the lower test RSS. The cubic model may reduce bias by fitting a nonlinear relationship better, but its extra flexibility can also increase variance.